<a href="https://colab.research.google.com/github/SangwanJeon/sensex-corpus-workflow-public/blob/main/Toolkit_Twitch_tuto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ensure that "Runtime", from the dropdown menu, is GPU.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Install the necessary repositories (to be performed once only)
**Don't restart if you get the message "WARNING: The following packages were previously imported in this runtime: [pydevd_plugins]". Just click "Cancel".**

In [ ]:
!pip uninstall -y torch torchaudio torchvision transformers whisperx accelerate -q
!pip cache purge -q

!pip install torch==2.3.1 torchaudio==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.41.2 -q
!pip install whisperx==3.3.1 -q
!pip install pandas yt-dlp webvtt-py -q

print("installation complete")

**IMPORTANT: Restart the runtime and then continue.**

In [ ]:
import torch
import torchvision
import transformers

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import HTML
import yt_dlp
import glob
import re
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 2. Downloading data from Twitch

A patched version of yt-dlp must be installed:

In [ ]:
!pip uninstall yt-dlp -y
!git clone https://github.com/mpeter50/yt-dlp
%cd yt-dlp
!pip install -r requirements.txt
!pip install -e .
!git checkout twitchvod-livechat

This retrieves the audio from a Twitch VoD:

In [ ]:
!yt-dlp -S "+size,+wa" --extract-audio 'https://www.twitch.tv/videos/2713870873' -o "/content/%(uploader)s/%(title)s.f%(format_id)s.%(ext)s" --extractor-args Twitch:device_id=did;client_integrity=cit1

To get the Twitch chat transcript, we will use the TwitchDowloaderCLI:

In [ ]:
!wget https://github.com/lay295/TwitchDownloader/releases/download/1.56.2/TwitchDownloaderCLI-1.56.2-Linux-x64.zip
!unzip TwitchDownloaderCLI-1.56.2-Linux-x64.zip
!sudo chmod +x TwitchDownloaderCLI

This retrieves the chat from a Twitch VoD:

In [ ]:
!./TwitchDownloaderCLI chatdownload --id 2713870873 -o /content/out.html

In [ ]:
from bs4 import BeautifulSoup

def parse_twitch_comments(file_path):
    """
    Parses a Twitch chat log HTML file and returns a DataFrame with embedded images for badges and emotes.

    Parameters:
        file_path (str): The path to the input HTML file.

    Returns:
        pd.DataFrame: A DataFrame containing 'time', 'author', and 'message'.
    """
    # Read the HTML content from the file
    with open(file_path, "r", encoding="utf-8") as file:
        html_data = file.read()

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(html_data, 'html.parser')

    # List to hold the parsed data
    data = []

    # Extract information from each 'pre' element
    for comment in soup.find_all('pre', class_='comment-root'):
        time = comment.text.split(']')[0].strip('[')  # Extract timestamp

        # Find the badge image (if exists)
        badge_img_tag = comment.find('img', class_='badge-image')
        badge_img = badge_img_tag['src'] if badge_img_tag else None  # Extract image URL

        # Extract author name
        author = comment.find('span', class_='comment-author').text if comment.find('span', class_='comment-author') else None

        # Create author column combining username and badge image
        if badge_img:
            author = f'{author} <img src="{badge_img}" width="20">'  # Append badge image next to author name

        # Extract comment message
        message = comment.find('span', class_='comment-message').text.strip(': ') if comment.find('span', class_='comment-message') else ""

        # Find and append emote images within the message
        for emote_img_tag in comment.find_all('img', class_='emote-image'):
            emote_img_url = emote_img_tag['src']
            emote_img_tag = f'<img src="{emote_img_url}" width="20">'
            message += f' {emote_img_tag}'  # Append emote image within the message

        # Append the extracted data to the list
        data.append([time, author, message])

    # Create a DataFrame
    df = pd.DataFrame(data, columns=['time', 'author', 'message'])

    # Convert the DataFrame to HTML and display the rendered images
    #return HTML(df.to_html(escape=False))
    return df
# Example usage:
twitch_df = parse_twitch_comments("/content/out.html")
# display(result)

In [ ]:
twitch_df

# 3. Transcription of the audio with WhisperX

In [ ]:
import whisperx
import torch
import glob
import pandas as pd
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

audio_file = glob.glob("/content/*/*.m4a")

model = whisperx.load_model(
    "small",
    device,
    compute_type="int8",
    language="fr"
)

audio = whisperx.load_audio(audio_file[0])

result = model.transcribe(
    audio,
    batch_size=2
)

del model
gc.collect()
torch.cuda.empty_cache()

model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=device
)

result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio,
    device,
    return_char_alignments=False
)

whisper_df = pd.DataFrame(result["segments"])
whisper_df1 = whisper_df[["start", "text"]]
whisper_df1.columns = ["time", "text"]

whisper_df1.head()


In [ ]:
whisper_df1

In [ ]:
def extract_content(input_df):
  extracted_messages = []
  for entry in input_df['replayChatItemAction']:
      #entry1 = json.loads(entry)
      actions = entry.get("actions", [])
      for action in actions:
          item = action.get("addChatItemAction", {}).get("item", {})
          renderer = item.get("liveChatTextMessageRenderer", {})
          message_runs = renderer.get("message", {}).get("runs", [])
          author_name = renderer.get("authorName", {}).get("simpleText", "")
          timestamp_usec = renderer.get("timestampText", {}).get("simpleText", "")
          badge_url = ''

          try:
              # Get author badges (if any)
              badges = renderer.get("authorBadges", [])

              # Loop through the badges and find the 32px thumbnail
              for badge in badges:
                  badge_thumbnails = badge.get('liveChatAuthorBadgeRenderer', {}).get('customThumbnail', {}).get('thumbnails', [])
                  for thumbnail in badge_thumbnails:
                      if thumbnail.get('width') == 32:  # Check if it's a 32-pixel image
                          badge_url = thumbnail.get('url', 'No badge')
                          break
          except (KeyError, IndexError):
              badge_url = ''
          if badge_url != '':
            author_name = author_name + ' <img src=' + badge_url + '>'

          # Initialize message content
          message_content = []
          for run in message_runs:
              if "text" in run:
                  message_content.append(run["text"])
              elif "emoji" in run:
                  emoji_info = run["emoji"]

                  if emoji_info.get("isCustomEmoji", False):
                      # YouTube/Twitch custom emotes → always images
                      thumbnails = emoji_info.get("image", {}).get("thumbnails", [])
                      if thumbnails:
                          emoji_image = thumbnails[-1].get("url", "")
                          message_content.append(f'<img src="{emoji_image}" alt="custom emoji">')
                  else:
                      # Normal Unicode emoji → keep the emoji itself
                      emoji_unicode = emoji_info.get("emojiId", "")
                      if emoji_unicode:
                          message_content.append(emoji_unicode)
                      else:
                          # Fallback: use the accessibility label
                          emoji_label = emoji_info.get("image", {}).get("accessibility", {}).get("accessibilityData", {}).get("label", "")
                          message_content.append(f":{emoji_label}:")

          # Join all parts of the message (text and emojis)
          full_message = "".join(message_content)

          # Append the extracted data to the list
          extracted_messages.append({
              "timestamp_usec": timestamp_usec,
              "author": author_name,
              "message": full_message
          })
  return extracted_messages


In [ ]:
def string_to_timedelta(time_str):
    if not pd.isnull(time_str):
        # Split the time string into hours, minutes, and seconds
        if time_str.startswith("-"):
            time_str = time_str.replace("-","")
            parts = time_str.split(':')
            if len(parts) == 3:
                hours = int(parts[0])
                minutes = int(parts[1])
                seconds = float(parts[2])
            elif len(parts) == 2:
                hours = 0
                minutes = int(parts[0])
                seconds = float(parts[1])
            else:
                return None

            # Convert hours, minutes, and seconds to timedelta
            time_delta = -timedelta(hours=hours, minutes=minutes, seconds=seconds).total_seconds()
        else:
            parts = time_str.split(':')
            if len(parts) == 3:
                hours = int(parts[0])
                minutes = int(parts[1])
                seconds = float(parts[2])
            elif len(parts) == 2:
                hours = 0
                minutes = int(parts[0])
                seconds = float(parts[1])
            else:
                return None

            # Convert hours, minutes, and seconds to timedelta
            time_delta = timedelta(hours=hours, minutes=minutes, seconds=seconds).total_seconds()

        return time_delta

# 4. Merge the chat and transcript data

Merge:

In [ ]:
twitch_df["time"] = twitch_df["time"].apply(lambda x: string_to_timedelta(str(x)))

In [ ]:
merged_df = pd.merge_ordered(whisper_df1, twitch_df, on='time')
merged_df = merged_df[merged_df["time"].notnull()].sort_values(by="time")

Save data to HTML:

In [ ]:
# Save DataFrame to HTML
merged_df1 = merged_df.fillna("")
merged_df1.to_html("/content/output.html", index=False, render_links=True, escape=False)

# HTML Content to add (Bootstrap + Custom Styles + JavaScript for sortable table)
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Document</title>
<link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
<style>
  body {
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 0;
  }
  .table-wrapper {
    padding: 20px;
    max-width: 100%;
    overflow-x: auto; /* Allow horizontal scrolling if necessary */
  }
  table {
    width: 100%;
    border-collapse: collapse;
    border: 1px solid #ddd;
    table-layout: fixed; /* Ensure columns have a predictable width */
  }
  th, td {
    border: 1px solid #ddd;
    padding: 8px;
    text-align: left;
    word-wrap: break-word; /* Ensure text wraps within the column */
    white-space: normal; /* Enable text wrapping */
    overflow: hidden; /* Prevent overflow of text */
  }
  th:first-child, td:first-child {
    width: 100px; /* Restrict the width of the first column */
  }
  th {
    position: sticky; /* Sticky headers */
    top: 0;
    background-color: #343a40; /* Dark header */
    color: white;
  }
  tr:nth-child(even) {
    background-color: #f2f2f2; /* Stripe effect for rows */
  }
</style>
<script src="https://code.jquery.com/jquery-3.5.1.slim.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/sticky-table-headers/js/jquery.stickytableheaders.min.js"></script>
</head>
<body>
<div class="table-wrapper">
<table border="1" class="dataframe table table-striped">
  <thead class="thead-dark">
    <tr style="text-align: left;">
"""

# Read the original HTML file
with open("/content/output.html", "r") as file:
    original_html = file.read()


# Replace the first part of the HTML with our header and add the original table
updated_html = html_content + original_html.replace('<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">',
                     '<table border="1" class="dataframe table table-striped">\n <thead class="thead-dark">\n <tr style="text-align: left;">') + """
</div>
<script>
  $(document).ready(function() {
    $("table").stickyTableHeaders();
  });
</script>
</body>
</html>
"""

# Write the modified HTML back to the file
with open("/content/output.html", "w") as file:
    file.write(updated_html)

In [ ]:
mer2 = merged_df.groupby(['time'])\
    .agg(chat=('message','count'),
         speech=('text','count')).reset_index()

mer2['time'] = (mer2['time'] / 60).astype(int)

# Group by the new time column and sum the other columns
result_df = mer2.groupby('time').sum().reset_index()
result_df2 = pd.melt(result_df, id_vars='time', value_vars=['chat', 'speech'])
result_df2.columns = ["time", "mode","value"]

In [ ]:
merged_df

# 5. Additional tools

Chat and speech density:

In [ ]:
sns.set_theme(style="darkgrid")
sns.set_context("talk")
plt.figure(figsize=(12, 6))
ax = sns.lineplot(data=result_df2, x='time', y='value', hue='mode')
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
plt.xlabel('Time in minutes')
plt.ylabel('Message Density')
plt.title('Message Density Over Time')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("message_density.png",dpi=600)
plt.show()

Get the video

In [ ]:
!./TwitchDownloaderCLI videodownload --id 2700984575 -o /content/video.mp4

Get the chat (.txt)

In [ ]:
!./TwitchDownloaderCLI chatdownload --id 2700984575 --timestamp-format Relative -o /content/chat.txt

Get an extract - of the video

In [ ]:
!./TwitchDownloaderCLI videodownload --id 2713672903 -b 07:50:54 -e 07:52:03 -o /content/video.mp4

Get an extract - of the audio

In [ ]:
!yt-dlp --download-sections "*07:50:54-07:52:03" -S "+size,+wa" --extract-audio 'https://www.twitch.tv/videos/2713672903' -o "/content/%(uploader)s/%(title)s.f%(format_id)s.%(ext)s" --extractor-args Twitch:device_id=did;client_integrity=cit1

Get an extract - of the chat

In [ ]:
!./TwitchDownloaderCLI chatdownload --id 2713672903 -b 07:50:54 -e 07:52:03 -o /content/out.html

Pseudonymisation of users based on roles

In [ ]:
import re
from bs4 import BeautifulSoup

def anonymize_html(file_path, output_path):
    # Read the HTML file
    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    # Badges URLs to identifie roles
    MODERATOR_BADGE = "https://static-cdn.jtvnw.net/badges/v1/3267646d-33f0-4b17-b3df-f923a41db1d0/2"
    BROADCASTER_BADGE = "https://static-cdn.jtvnw.net/badges/v1/5527c58c-fb7d-422d-b71b-f309dcb85cc1/2"

    # Mappage of pseudo pseudonomised
    user_mapping = {}
    moderator_counter = 1
    user_counter = 1

    rows = soup.find_all("tr")[1:]

    for row in rows:
        columns = row.find_all("td")
        if len(columns) >= 4:
            author_td = columns[2]
            message_td = columns[3]

            author_text = author_td.get_text(strip=True)
            author_lower = author_text.lower()

            # Determine the role
            role = "User"
            img_tag = author_td.find("img")
            if img_tag:
                badge_url = img_tag["src"]
                if badge_url == BROADCASTER_BADGE:
                    role = "Broadcaster"
                elif badge_url == MODERATOR_BADGE:
                    role = "Moderator"

            # Pseudonymisation
            if author_lower and author_lower not in user_mapping:
                if role == "Broadcaster":
                    user_mapping[author_lower] = "Broadcaster"
                elif role == "Moderator":
                    user_mapping[author_lower] = f"Moderator{moderator_counter}"
                    moderator_counter += 1
                else:
                    user_mapping[author_lower] = f"User{user_counter}"
                    user_counter += 1

            # Replace in the author column
            if author_lower in user_mapping:
                author_td.clear()
                author_td.string = user_mapping[author_lower]

    # Create the regex once
    if user_mapping:
        pattern = re.compile(
            r"@(" + "|".join(re.escape(name) for name in user_mapping.keys()) + r")",
            flags=re.IGNORECASE
        )

        def replace_tag(match):
            name = match.group(1).lower()
            anon_name = user_mapping.get(name, name)
            return f"@{anon_name}"

        # Apply the replacement in all messages
        for row in rows:
            columns = row.find_all("td")
            if len(columns) >= 4:
                message_td = columns[3]
                if message_td and message_td.get_text(strip=True):
                    message_text = message_td.decode_contents()
                    updated = pattern.sub(replace_tag, message_text)
                    message_td.clear()
                    message_td.append(BeautifulSoup(updated, "html.parser"))

    # Save
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(str(soup))

    print("pseudonymisation completed")

# Example of using
anonymize_html("/content/output.html", "/content/output_pseudonymised.html")